In [ ]:
# Lab type: debug
# Course: AI403 — Building Production AI Agents
# Lesson: The Agent Loop: Plan, Act, Observe, Decide
# Task: The agent loop below never stops when a tool fails. There are 3 bugs.
# After fixing each bug, write a one-sentence explanation in the comment cell below it.

# Lab: The Loop That Wouldn't Stop

**How these labs work.** The "model" here is `scripted_model`, a plain Python function that makes the same choices an LLM would in this situation. That keeps every run deterministic and free, so each bug reproduces exactly. The harness around it — the loop, the tools, the checks — is real code, and it's what you'll be fixing.

**Outputs are cleared.** Run every cell top to bottom.

## Setup

In [ ]:
# No installs needed: this lab uses the Python standard library only.

In [ ]:
# The Nimbus Analytics billing service, with a switch to simulate an outage
ORDERS = {"ORD-881": {"amount_cents": 4000, "status": "paid"},
          "ORD-904": {"amount_cents": 230000, "status": "paid"}}
SERVICE = {"up": True, "calls": 0}

class EmergencyBrake(RuntimeError):
    """Stands in for the on-call engineer killing a runaway process."""

def get_order(order_id):
    SERVICE["calls"] += 1
    if SERVICE["calls"] > 50:
        raise EmergencyBrake("50 calls to the billing service in one run - process killed")
    if not SERVICE["up"]:
        return "Error: billing service timeout"
    order = ORDERS.get(order_id)
    return f"order {order_id}: amount {order['amount_cents']} cents, {order['status']}" if order else "not found"

def scripted_model(messages):
    """Stand-in for an LLM: looks the order up, retrying until it sees an amount."""
    last = messages[-1]
    if last["role"] == "tool" and "amount" in str(last["content"]):
        return {"type": "final", "text": f"Found it - {last['content']}"}
    return {"type": "tool", "tool": "get_order", "args": {"order_id": "ORD-881"}}

TOOLS = {"get_order": get_order}
print(get_order("ORD-881"))

## The loop under review

In [ ]:
# --- THE AGENT LOOP (review this code — is it correct?) ---
def run_agent(model, tools, task):
    messages = [{"role": "user", "content": task}]
    while True:                                                    # <- look closely
        action = model(messages)
        if action["type"] == "final":
            return {"status": "done", "answer": action["text"]}
        result = tools[action["tool"]](**action["args"])           # <- and at what comes back
        messages.append({"role": "tool", "content": result})

def run_safely(task):
    """Resets the service counter and reports what happened."""
    SERVICE["calls"] = 0
    try:
        return run_agent(scripted_model, TOOLS, task)
    except EmergencyBrake as e:
        return {"status": "KILLED", "reason": str(e)}

SERVICE["up"] = True
print("healthy:", run_safely("What is the amount on ORD-881?"))
SERVICE["up"] = False
print("outage: ", run_safely("What is the amount on ORD-881?"))

**Question 1.** On the healthy path the loop works. During the outage it only stops because the emergency brake kills it after 50 calls. Explain, in terms of *what the model sees*, why the model keeps calling `get_order` — and why that's reasonable behaviour for a model.

In [ ]:
# Your explanation (as a comment):

<details>
<summary>🔑 Reveal answer — Question 1</summary>

The tool reports the failure as an ordinary string. To the model, `"Error: billing service timeout"` is just a result without an amount in it, so the task isn't done and trying again is the helpful thing to do. Nothing tells it the failure won't clear up, and nothing in the loop counts the attempts. A real LLM might give up after a few tries or might not; either way, stopping is left to the model's judgement — which is the defect.

</details>

## Bug 1: the tool hides its failure

**Question 2.** Rewrite `get_order` so every result is structured: `{"ok": True, ...}` on success and `{"ok": False, "error": ..., "retryable": ...}` on failure. Keep the emergency brake. Update `scripted_model` so it looks for `amount_cents` in a successful result.

In [ ]:
# Work here: structured get_order (and the matching scripted_model)

In [ ]:
# Bug 1 explanation (one sentence):

<details>
<summary>🔑 Reveal answer — Question 2</summary>

```python
def get_order(order_id):
    SERVICE["calls"] += 1
    if SERVICE["calls"] > 50:
        raise EmergencyBrake("50 calls to the billing service in one run - process killed")
    if not SERVICE["up"]:
        return {"ok": False, "error": "billing_service_timeout", "retryable": True}
    order = ORDERS.get(order_id)
    if order is None:
        return {"ok": False, "error": "order_not_found", "retryable": False}
    return {"ok": True, "order_id": order_id, **order}

def scripted_model(messages):
    last = messages[-1]
    if last["role"] == "tool" and last["content"].get("ok") and "amount_cents" in last["content"]:
        return {"type": "final", "text": f"Found it - {last['content']['amount_cents']} cents"}
    return {"type": "tool", "tool": "get_order", "args": {"order_id": "ORD-881"}}

TOOLS = {"get_order": get_order}
```

*Explanation:* a failure returned as text is indistinguishable from data, so neither the harness nor the model can act on it; a structured error with `retryable` can be counted and reasoned about.

</details>

## Bugs 2 and 3: the loop has no exits

**Question 3.** Rewrite `run_agent` with:

- a hard `max_steps` (default 8) that returns `{"status": "step_limit", "answer": None}`;
- a `max_consecutive_errors` (default 2) that returns `{"status": "tool_failing", "error": ...}`;
- repetition detection: the same tool with the same arguments 3 times in a row returns `{"status": "no_progress"}`.

No exit except `done` may return an answer. Then re-run `run_safely` for the healthy and outage cases.

In [ ]:
# Work here: bounded run_agent

In [ ]:
# Bug 2 explanation (no cap):
# Bug 3 explanation (errors never end the run):

<details>
<summary>🔑 Reveal answer — Question 3</summary>

```python
def run_agent(model, tools, task, max_steps=8, max_consecutive_errors=2, max_repeats=3):
    messages = [{"role": "user", "content": task}]
    errors, recent = 0, []
    for step in range(1, max_steps + 1):
        action = model(messages)
        if action["type"] == "final":
            return {"status": "done", "answer": action["text"], "steps": step}
        signature = (action["tool"], tuple(sorted(action["args"].items())))
        recent = (recent + [signature])[-max_repeats:]
        result = tools[action["tool"]](**action["args"])
        errors = errors + 1 if not result.get("ok", True) else 0
        if errors >= max_consecutive_errors:
            return {"status": "tool_failing", "error": result["error"], "answer": None, "steps": step}
        if len(recent) == max_repeats and len(set(recent)) == 1:
            return {"status": "no_progress", "answer": None, "steps": step}
        messages.append({"role": "tool", "content": result})
    return {"status": "step_limit", "answer": None, "steps": max_steps}

SERVICE["up"] = True
print("healthy:", run_safely("What is the amount on ORD-881?"))
SERVICE["up"] = False
print("outage: ", run_safely("What is the amount on ORD-881?"))
```

The outage now ends after 2 calls with `tool_failing` and the error name, instead of 50 calls and a killed process. *Bug 2:* with no step cap, stopping depends on the model. *Bug 3:* errors never ended the run, so a failing dependency turned into an unbounded retry loop.

</details>

**Question 4.** Tests make the exits permanent. Write three asserts: healthy → `done`; outage → `tool_failing` within 2 calls; and a model that always asks for the same *successful* lookup but never answers → `no_progress` (write a tiny `stubborn_model` for it).

In [ ]:
# Work here: three asserts

<details>
<summary>🔑 Reveal answer — Question 4</summary>

```python
SERVICE["up"] = True
assert run_safely("What is the amount on ORD-881?")["status"] == "done"

SERVICE["up"] = False
out = run_safely("What is the amount on ORD-881?")
assert out["status"] == "tool_failing" and SERVICE["calls"] == 2, out

def stubborn_model(messages):
    return {"type": "tool", "tool": "get_order", "args": {"order_id": "ORD-904"}}

SERVICE["up"], SERVICE["calls"] = True, 0
out = run_agent(stubborn_model, TOOLS, "Amount on ORD-904?")
assert out["status"] == "no_progress" and out["answer"] is None, out
print("all exits behave")
```

</details>

## Summary

1. A tool failure returned as ordinary text looks like _______ to both the model and the harness.
2. Step caps, error thresholds and repetition checks are enforced by the _______, not the model.
3. Every exit except `done` returns a status and _______ answer.

<details>
<summary>🔑 Reveal summary answers</summary>

1. **data**
2. **harness (the loop)**
3. **no**

</details>